Este projeto será para conscientizar a população sobre os casos alto de feminicidio lembrando que este projeto é para uso academico onde será desenvolvido o ELT e será criado um Dashbord completo com os principais temas deste assunto.

In [ ]:
# Biblioteca para manipulação de arquivos e diretórios
import os
import csv
from datetime import datetime
import pandas as pd
import numpy
from sqlalchemy import text
from db_config import get_engine
import pymysql

In [10]:
# Lendo o arquivo CSV e armazenando em um DataFrame

df_feminicidio = pd.read_csv('feminicidio_sintetico_2023_2026.csv', sep=',', encoding='utf-8')

In [ ]:
# Conta o número de valores nulos em cada coluna do DataFrame
# df_feminicidio.isnull().sum()

In [11]:
# Modificando name das culunas 

df_feminicidio.rename(columns={'municipio_cod':'cod_municipio','risp':'regiao_de_seguranca','rmbh':'regiao_metropolitana'}, inplace=True)

In [13]:
# tratamento das colunas 

# converter data
df_feminicidio['data_fato'] = pd.to_datetime(df_feminicidio['data_fato'], errors='coerce')

# garantir tipos numéricos
df_feminicidio['qtde_vitimas'] = pd.to_numeric(df_feminicidio['qtde_vitimas'], errors='coerce')

# remover nulos importantes
df_feminicidio = df_feminicidio.dropna(subset=['data_fato', 'qtde_vitimas'])

In [14]:
# colunas calculadas 


df_feminicidio['media_vitimas'] = df_feminicidio['qtde_vitimas'].mean()
df_feminicidio['maximo_vitimas'] = df_feminicidio['qtde_vitimas'].max()
df_feminicidio['minimo_vitimas'] = df_feminicidio['qtde_vitimas'].min()


# criando coluna Ano_mes 

df_feminicidio['ano_mes'] = df_feminicidio['data_fato'].dt.strftime('%Y/%m')

# ultima atualização do dataframe
df_feminicidio['data_execucao'] = datetime.now().strftime('%d/%m/%Y %H:%M:%S')

In [15]:
# Agora será criado A tabela Fato e as tabelas Dimensão para o modelo dimensional.


# =========================
# DIMENSÕES
# =========================

# Municipio
dim_municipio = df_feminicidio[['cod_municipio', 'municipio_fato']].drop_duplicates()

# Data
dim_data = df_feminicidio[['data_fato', 'mes', 'ano', 'ano_mes']].drop_duplicates()
dim_data['data_id'] = range(1, len(dim_data) + 1)

# Região
dim_regiao = df_feminicidio[['regiao_de_seguranca', 'regiao_metropolitana']].drop_duplicates()
dim_regiao['regiao_id'] = range(1, len(dim_regiao) + 1)

# Status
dim_status = df_feminicidio[['tentado_consumado']].drop_duplicates()
dim_status['status_id'] = range(1, len(dim_status) + 1)

# =========================
# FATO (JOIN COM IDS)
# =========================

# Merge com dimensões
fato = df_feminicidio.merge(dim_data, on=['data_fato','mes','ano','ano_mes'])
fato = fato.merge(dim_regiao, on=['regiao_de_seguranca','regiao_metropolitana'])
fato = fato.merge(dim_status, on='tentado_consumado')

# Selecionar colunas finais
fato = fato[
    [
        'cod_municipio',
        'data_id',
        'regiao_id',
        'status_id',
        'qtde_vitimas',
        'media_vitimas',
        'maximo_vitimas',
        'minimo_vitimas',
        'data_execucao'
    ]
]

In [22]:

engine = get_engine()

try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT 1"))
        print("✅ Banco respondeu:", result.scalar())
except Exception as e:
    print("❌ Erro:")
    print(e)

✅ Banco respondeu: 1


In [23]:
dim_municipio.to_sql('dim_municipio', engine, if_exists='replace', index=False)
dim_data.to_sql('dim_data', engine, if_exists='replace', index=False)
dim_regiao.to_sql('dim_regiao', engine, if_exists='replace', index=False)
dim_status.to_sql('dim_status', engine, if_exists='replace', index=False)

fato.to_sql('fato_crimes', engine, if_exists='replace', index=False)

3000